# Kaggle影像消融：I2：仅地形通道

DeepLabV3+-ResNet50、seed42、physical batch4、80 epochs；只改变有效输入通道，不读取Test指标。


In [ ]:
from pathlib import Path
import importlib.metadata, importlib.util, json, os, subprocess, sys

REPO_URL = 'https://github.com/song110585-cpu/lunar-linear.git'
REPO_BRANCH = 'test-new-module'
REQUIRED_COMMIT = '6d33bf0'
REPO_DIR = Path('/kaggle/working/lunar-linear')
PROJECT_DIR = REPO_DIR / 'LTL-Net'
OUTPUT_ROOT = Path('/kaggle/working')
CONFIG_NAME = 'v6_overlap40_deeplab_input_terrain_only_batch4_seed42.json'
EXPECTED_MODE = 'terrain_only'

DATA_CANDIDATES = [
    Path('/kaggle/input/datasets/yuanssy/datav6-overlap40/dataset_v6_random811_overlap40'),
    Path('/kaggle/input/datasets/changyasong/datav6-overlap40/dataset_v6_random811_overlap40'),
    Path('/kaggle/input/datasets/changyasong/v6data/dataset_v6_random811_overlap40'),
    Path('/kaggle/input/datav6-overlap40/dataset_v6_random811_overlap40'),
    Path('/kaggle/input/v6data/dataset_v6_random811_overlap40'),
]


## 1. 环境与代码


In [ ]:
required = [('rasterio', 'rasterio'), ('tqdm', 'tqdm')]
missing = [package for module, package in required if importlib.util.find_spec(module) is None]
try:
    smp_version = importlib.metadata.version('segmentation-models-pytorch')
except importlib.metadata.PackageNotFoundError:
    smp_version = None
if smp_version != '0.5.0':
    missing.append('segmentation-models-pytorch==0.5.0')
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])

if not REPO_DIR.exists():
    subprocess.check_call(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)])
elif not (REPO_DIR / '.git').is_dir():
    raise RuntimeError(f'目录存在但不是Git仓库: {REPO_DIR}')
else:
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', REPO_BRANCH])
subprocess.check_call(['git', '-C', str(REPO_DIR), 'merge-base', '--is-ancestor', REQUIRED_COMMIT, 'HEAD'])
commit = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'], text=True).strip()
print('Git commit:', commit)
subprocess.run(['nvidia-smi'], check=False)


## 2. 自动定位并核验Kaggle数据


In [ ]:
discovered = sorted({
    path for path in Path('/kaggle/input').rglob('dataset_v6_random811_overlap40')
    if path.is_dir()
})
ordered = []
for path in [*DATA_CANDIDATES, *discovered]:
    if path not in ordered and (path / 'dataset_protocol.json').is_file():
        ordered.append(path)
assert ordered, '未找到dataset_v6_random811_overlap40；请先用Add Input挂载数据集。'
DATA_ROOT = ordered[0]
CONFIG_PATH = PROJECT_DIR / 'configs' / CONFIG_NAME
config = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
assert config['channel_mode'] == EXPECTED_MODE, config['channel_mode']
assert config['seed'] == 42 and config['epochs'] == 80
assert config['batch_size'] == 4 and config['accum_steps'] == 1
assert config['automatic_test_evaluation'] is False
print('数据目录:', DATA_ROOT)
print('实验:', config['run_name'])
print('channel_mode:', config['channel_mode'])
print('输出:', OUTPUT_ROOT / f"result_{config['run_name']}")


## 3. 开始训练


In [ ]:
command = [
    sys.executable, str(PROJECT_DIR / 'scripts/run_autodl_channel_ablation.py'),
    '--project-dir', str(PROJECT_DIR),
    '--config', str(CONFIG_PATH),
    '--data-dir', str(DATA_ROOT),
    '--output-dir', str(OUTPUT_ROOT),
]
env = os.environ.copy(); env['PYTHONUNBUFFERED'] = '1'
print(' '.join(command), flush=True)
subprocess.check_call(command, cwd=PROJECT_DIR, env=env)


## 4. 结果检查与下载


In [ ]:
result_dir = OUTPUT_ROOT / f"result_{config['run_name']}"
metrics_path = result_dir / 'metrics.json'
archive_path = Path(str(result_dir) + '.zip')
assert metrics_path.is_file(), metrics_path
assert archive_path.is_file(), archive_path
metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
assert metrics['test_evaluated'] is False
print(json.dumps(metrics, ensure_ascii=False, indent=2))
print('下载:', archive_path)
